<a href="https://colab.research.google.com/github/Suheilee/BLEVEPressurePrediction/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Suheilee Amashki Samarasinghe
22187972
Module: Machine Learning

This section sets up the drive in order to access the training and testing csv files, and loads the data.

In [ ]:
#Set up drive to access training and test data
from google.colab import drive
drive.mount('/content/drive')

import os
folder_path = '/content/drive/MyDrive/ML'   # Change if folder name/path is different

# Load data
train = pd.read_csv(f'{folder_path}/train.csv')
test = pd.read_csv(f'{folder_path}/test.csv')
sample = pd.read_csv(f'{folder_path}/sample_prediction.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Target column:", 'Target Pressure (bar)' in train.columns)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train shape: (10050, 25)
Test shape: (3203, 24)
Target column: True


This section loads the libraries used within the notebook. A few additional libraries are also loaded, as a result of previous explorations.

For example, I experimented with a catboost model before removing the model due to operational limitations.

Unimportant warnings are also suppressd to increase the readability of the output.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GroupKFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

Libraries loaded.


After loading all the necessary libraries, this section will delete any rows with missing target values.

10 such rows were identified and deleted.

In [ ]:
# Drop rows with missing target (if any)
train = train.dropna(subset=[target])
print("After dropping missing target:", train.shape)

Train shape: (10050, 25)
Test shape: (3203, 24)
Target column present: True
After dropping missing target: (10040, 25)


Feature Engineering: I have created a few additonal features to better help train the machine learning models, rather than depending solely on the raw data.

The same transformations are applied consistently to both the training and testing data sets.

In [ ]:
def add_features(df):
    """Add engineered features to the dataframe."""
    #Created a copy to avoid modifying the original dataframe
    df = df.copy()

    # Basic features from previously tested models
    df['Tank_Volume'] = df['Tank Width (m)'] * df['Tank Length (m)'] * df['Tank Height (m)']
    df['Vapour_Vol_Ratio'] = df['Vapour Height (m)'] / (df['Tank Height (m)'] + 1e-8)
    df['Inv_Dist'] = 1 / (df['Obstacle Distance to BLEVE (m)'] + 1e-8)
    df['Sensor_XY_Dist'] = np.sqrt(df['Sensor Position x']**2 + df['Sensor Position y']**2)
    df['Temp_Diff'] = df['Vapour Temperature (K)'] - df['Liquid Temperature (K)']
    df['Crit_Pressure_Ratio'] = df['Tank Failure Pressure (bar)'] / (df['Liquid Critical Pressure (bar)'] + 1e-8)

    # Additional physics-based features
    df['Pressure_Volume'] = df['Tank Failure Pressure (bar)'] * df['Tank_Volume']
    df['Liq_Vap_Ratio'] = (df['Liquid Ratio (%)'] + 1e-8) / (df['Vapour Height (m)'] + 1e-8)
    df['Temp_Diff_sq'] = df['Temp_Diff'] ** 2
    df['Dist_Height_Interaction'] = df['Obstacle Distance to BLEVE (m)'] * df['Obstacle Height (m)']
    df['Sensor_z_rel'] = df['Sensor Position z'] / (df['BLEVE Height (m)'] + 1e-8)

    return df

# Apply to train and test
X_train_raw = train.drop(columns=[target, 'ID'])
X_test_raw = test.drop(columns=['ID'])

X_train = add_features(X_train_raw)
X_test = add_features(X_test_raw)

print("Train features shape after engineering:", X_train.shape)
print("Test features shape after engineering:", X_test.shape)

Train features shape after engineering: (10040, 34)
Test features shape after engineering: (3203, 34)


Missing Values:
Missing numerical values are replaced using the median of each column calculated from the training dataset.

Missing categorical values are replaced using the mode (most frequent category).

This is done to ensure all the models will receive complete data, without any null values.

In [ ]:
# Identify numeric and categorical columns
numeric_cols = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", categorical_cols)

# Fill numeric missing values with median (from train only)
for col in numeric_cols:
    median_val = X_train[col].median()
    X_train[col].fillna(median_val, inplace=True)
    X_test[col].fillna(median_val, inplace=True)

# Fill categorical missing values with mode (most frequent)
for col in categorical_cols:
    mode_val = X_train[col].mode()[0]
    X_train[col].fillna(mode_val, inplace=True)
    X_test[col].fillna(mode_val, inplace=True)

print("Missing values handled.")

Numeric columns: 33
Categorical columns: ['Status']
Missing values handled.


One hot encoding for categorical Columns:
The 'status' feature is converted into numerical form to ensure that the models will be able to process it.

In [ ]:
# One-hot encode categorical columns
X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Align test columns to train (add missing, drop extra)
for col in X_train.columns:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[X_train.columns]

print("Final train shape:", X_train.shape)
print("Final test shape:", X_test.shape)

Final train shape: (10040, 40)
Final test shape: (3203, 40)


Prepare target:
The target variable also undergoes preprocessing to improve model stability and predictive performance.

This is done through Winsorization and Log Transformation.

The predictions are later converted back to the original scale with the exponential inverse transformation.

In [ ]:
from scipy.stats.mstats import winsorize

# Original target
y_raw = train[target].values

# Winsorize at 1st and 99th percentiles (caps extreme values)
y_winsorized = winsorize(y_raw, limits=(0.01, 0.01))

# Log transform (log(1+p)) for better modeling
y_log = np.log1p(y_winsorized)

print("Target transformation done.")
print(f"Original target range: [{y_raw.min():.2f}, {y_raw.max():.2f}]")
print(f"Winsorized range: [{y_winsorized.min():.2f}, {y_winsorized.max():.2f}]")

Target transformation done.
Original target range: [0.02, 9.17]
Winsorized range: [0.03, 2.44]


Train - Validation Split:
I've split the data set with 80% training data, and 20% validation data.
This is used to evaluate the model performance while developing my models.

In [ ]:
X_train_split, X_val, y_train_log, y_val_log = train_test_split(
    X_train, y_log, test_size=0.2, random_state=42
)

print("Train size:", X_train_split.shape[0])
print("Validation size:", X_val.shape[0])

Train size: 8032
Validation size: 2008


Model Evaluation Function

This function was created as a helper function to simplify the model comparison process.

In [ ]:
def evaluate_model(model, X_train, y_train, X_val, y_val, model_name):
    """Train a model and return validation MAPE and R2."""
    model.fit(X_train, y_train)
    y_pred_log = model.predict(X_val)
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_val)

    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name:15s} | MAPE: {mape:.4f} | R2: {r2:.4f}")
    return mape, r2, model

Hyperparameter tuning with KFold

In [ ]:
# Extract sensor IDs for grouping
groups = train['Sensor ID'].values[:len(X_train)]  # same order as X_train

# Define GroupKFold
gkf = GroupKFold(n_splits=5)

Tune XGBoost

In [ ]:
print("\n=== Fast Tuning XGBoost (Boosting) ===")
# Use subset for tuning
X_sub, _, y_sub, _, groups_sub, _ = train_test_split(
    X_train, y_log, groups, train_size=5000, random_state=42
)

xgb_base = xgb.XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist')
param_dist = {
    'n_estimators': [500, 700],
    'max_depth': [6, 8],
    'learning_rate': [0.03, 0.05],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9],
    'reg_alpha': [0, 0.01],
    'reg_lambda': [1, 1.5]
}
search_xgb = RandomizedSearchCV(xgb_base, param_dist, n_iter=8, cv=3,
                                scoring='neg_mean_absolute_percentage_error',
                                random_state=42, n_jobs=-1, verbose=1)
search_xgb.fit(X_sub, y_sub, groups=groups_sub)
best_xgb = search_xgb.best_estimator_
print("Best XGBoost params:", search_xgb.best_params_)


=== Fast Tuning XGBoost (Boosting) ===
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best XGBoost params: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 700, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 0.9}


Train Random Forest

In [ ]:
print("\nTraining Random Forest")
from sklearn.ensemble import RandomForestRegressor

# Random Forest with good defaults (300 trees, depth 20)
rf = RandomForestRegressor(n_estimators=300, max_depth=20,
                           min_samples_split=5, random_state=42, n_jobs=-1)
# Fit on full training data directly (no tuning needed)
rf.fit(X_train, y_log)
print("Random Forest trained.")


Training Random Forest
Random Forest trained.


Ridge Regression Model

In [ ]:
print("\nTuning Ridge Regression")
from sklearn.linear_model import RidgeCV

# RidgeCV performs efficient cross-validation over alphas
alphas = [0.01, 0.1, 1, 10, 50, 100]
ridge = RidgeCV(alphas=alphas, scoring='neg_mean_absolute_percentage_error', cv=5)
ridge.fit(X_train, y_log)
best_ridge = ridge
print(f"Best Ridge alpha: {ridge.alpha_}")


Tuning Ridge Regression
Best Ridge alpha: 100.0


LightGBM Model

In [ ]:
# ==================== FAST LIGHTGBM TUNING (10–15 min) ====================
print("\n=== Fast Tuning LightGBM ===")

lgb_fast = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)

param_dist_lgb = {
    'n_estimators': [500, 700, 900],
    'max_depth': [6, 8, 10, -1],
    'learning_rate': [0.03, 0.05, 0.07],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.8, 0.9],
    'reg_alpha': [0, 0.01],
    'reg_lambda': [1, 1.5]
}

random_search_lgb = RandomizedSearchCV(
    lgb_fast, param_dist_lgb, n_iter=12, cv=3,
    scoring='neg_mean_absolute_percentage_error',
    random_state=42, n_jobs=-1, verbose=1
)
random_search_lgb.fit(X_sub, y_sub, groups=groups_sub)

best_lgb = random_search_lgb.best_estimator_
print("Best LightGBM params:", random_search_lgb.best_params_)


=== Fast Tuning LightGBM ===
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best LightGBM params: {'subsample': 0.8, 'reg_lambda': 1.5, 'reg_alpha': 0.01, 'n_estimators': 700, 'max_depth': -1, 'learning_rate': 0.05, 'colsample_bytree': 0.9}


Validation Performance Comparison

In [ ]:
# Create validation split
X_tr, X_val, y_tr_log, y_val_log = train_test_split(X_train, y_log, test_size=0.2, random_state=42)

# Retrain XGBoost on training split
best_xgb.fit(X_tr, y_tr_log)

# Retrain LightGBM on training split
best_lgb.fit(X_tr, y_tr_log)

# Retrain Random Forest on training split
rf.fit(X_tr, y_tr_log)

# Ridge: refit on training split using the best alpha from earlier
from sklearn.linear_model import Ridge
ridge_val = Ridge(alpha=ridge.alpha_)
ridge_val.fit(X_tr, y_tr_log)

# Collect validation MAPE and store predictions
models = {
    'XGBoost': best_xgb,
    'LightGBM': best_lgb,
    'RandomForest': rf,
    'Ridge': ridge_val
}
val_mape = {}
val_preds_log = {}   # store predictions (log scale)

print("\n=== Individual Model Validation Performance ===")
for name, model in models.items():
    pred_log = model.predict(X_val)
    val_preds_log[name] = pred_log
    pred = np.expm1(pred_log)
    true = np.expm1(y_val_log)
    mape = mean_absolute_percentage_error(true, pred)
    r2 = r2_score(true, pred)
    val_mape[name] = mape
    print(f"{name:12s} | MAPE: {mape:.4f} | R2: {r2:.4f}")


=== Individual Model Validation Performance ===
XGBoost      | MAPE: 0.1260 | R2: 0.9723
LightGBM     | MAPE: 0.1221 | R2: 0.9744
RandomForest | MAPE: 0.1779 | R2: 0.9439
Ridge        | MAPE: 0.7213 | R2: 0.5713


Ensemble Weight optimisation and prediction csv creation.

In [ ]:
# Use finer grid search (0.02 increments)
best_mape = float('inf')
best_weights_fine = None
val_true_orig = np.expm1(y_val_log)

# Get current best weights as starting point (taken from previous grid searches)
current_weights = np.array([0.44, 0.28, 0.20, 0.08])

# Search in a ±0.1 range around each weight with step 0.02
for w_xgb in np.arange(max(0.1, current_weights[0]-0.1), min(0.7, current_weights[0]+0.1), 0.02):
    for w_lgb in np.arange(max(0.1, current_weights[1]-0.1), min(0.7, current_weights[1]+0.1), 0.02):
        for w_rf in np.arange(max(0.05, current_weights[2]-0.1), min(0.5, current_weights[2]+0.1), 0.02):
            w_ridge = 1 - (w_xgb + w_lgb + w_rf)
            if w_ridge < 0 or w_ridge > 0.2:
                continue
            weights = np.array([w_xgb, w_lgb, w_rf, w_ridge])
            ens_log = (weights[0] * val_preds_log['XGBoost'] +
                       weights[1] * val_preds_log['LightGBM'] +
                       weights[2] * val_preds_log['RandomForest'] +
                       weights[3] * val_preds_log['Ridge'])
            ens_pred = np.expm1(ens_log)
            mape = mean_absolute_percentage_error(val_true_orig, ens_pred)
            if mape < best_mape:
                best_mape = mape
                best_weights_fine = weights

print(f"Best MAPE from fine search: {best_mape:.5f}")
print("Best weights:", best_weights_fine)

# Use these weights to generate submission
if best_weights_fine is not None:
    test_pred_log = (best_weights_fine[0] * best_xgb.predict(X_test) +
                     best_weights_fine[1] * best_lgb.predict(X_test) +
                     best_weights_fine[2] * rf.predict(X_test) +
                     best_weights_fine[3] * ridge.predict(X_test))
    test_pred = np.expm1(test_pred_log)
    test_pred = np.clip(test_pred, 0, None)
    sub_fine = pd.DataFrame({'ID': test['ID'], target: test_pred})
    sub_fine.to_csv('prediction.csv', index=False)
    print("Submission saved: prediction.csv")

Best MAPE from fine search: 0.11972
Best weights: [0.5  0.36 0.14 0.  ]
Submission saved: prediction.csv
